In [1]:
import geopandas as gpd
import pandas as pd
import folium
import branca.colormap as cm

# Carregar shapefile
shp_path = "MT_setores_CD2022.shp"
gdf = gpd.read_file(shp_path)

# Carregar a segunda tabela (df) e contar quantas vezes cada setor aparece
df = pd.read_csv('C:/Users/mathg/OneDrive/Documents/Diretório/Water/VSE/setoresgrafico2.csv')  # Substitua pelo nome correto do arquivo
df = df[df['fraude']==1]
df["CD_SETOR"] = df["CD_SETOR"].astype(str).str.replace(".0", "", regex=False)
setor_counts = df["CD_SETOR"].value_counts().reset_index()
setor_counts.columns = ["CD_SETOR", "COUNT"]

# Verificar se setor_counts está correto
print(setor_counts.head())  # Deve exibir CD_SETOR e COUNT

# Converter para string e unir ao GeoDataFrame
gdf["CD_SETOR"] = gdf["CD_SETOR"].astype(str)
gdf_filtrado = gdf.merge(setor_counts, on="CD_SETOR", how="inner")

# Verificar se o merge funcionou
print(gdf_filtrado.head())  # Deve conter a coluna "COUNT"

# Criar um mapa centrado
if not gdf_filtrado.empty:
    centro_x = gdf_filtrado.geometry.centroid.x.mean()
    centro_y = gdf_filtrado.geometry.centroid.y.mean()
    location = [centro_y, centro_x]
else:
    raise ValueError("Nenhum setor correspondente encontrado.")

m = folium.Map(location=location, zoom_start=10)

# Criar um colormap (escala de cores)
if "COUNT" in gdf_filtrado.columns:
    colormap = cm.LinearColormap(
        colors=["yellow", "orange", "red"],  # Escala de cores de frio para quente
        vmin=gdf_filtrado["COUNT"].min(),
        vmax=gdf_filtrado["COUNT"].max()
    )
else:
    raise ValueError("Coluna 'COUNT' não encontrada após o merge.")

# Adicionar setores ao mapa
for _, row in gdf_filtrado.iterrows():
    color = colormap(row["COUNT"])  # Definir cor com base no número de observações
    folium.GeoJson(
        row.geometry,
        style_function=lambda x, color=color: {"fillColor": color, "color": "black", "weight": 1, "fillOpacity": 0.7},
        tooltip=f"Setor {row['CD_SETOR']}<br>Observações: {row['COUNT']}"
    ).add_to(m)

# Adicionar legenda
colormap.caption = "Número de Observações"
colormap.add_to(m)

# Salvar o mapa
m.save("mapa_interativo_colorido.html")

# Exibir no Jupyter Notebook (se necessário)
m


          CD_SETOR  COUNT
0  330455705290116     51
1  330455705210430     24
2  330455705290117     24
3  330455705290039     18
4  411820405000199     17
          CD_SETOR SITUACAO CD_SIT CD_TIPO  AREA_KM2 CD_REGIAO     NM_REGIAO  \
0  510340305400040   Urbana      2       4  1.059818         5  Centro-Oeste   
1  510340305410008   Urbana      1       0  0.097850         5  Centro-Oeste   
2  510340305410015   Urbana      1       0  0.198125         5  Centro-Oeste   
3  510340305410025   Urbana      1       0  0.194092         5  Centro-Oeste   
4  510340305410030   Urbana      1       0  0.166905         5  Centro-Oeste   

  CD_UF        NM_UF   CD_MUN  ... CD_AGLOM NM_AGLOM CD_RGINT NM_RGINT  \
0    51  Mato Grosso  5103403  ...     None     None     5101   Cuiabá   
1    51  Mato Grosso  5103403  ...     None     None     5101   Cuiabá   
2    51  Mato Grosso  5103403  ...     None     None     5101   Cuiabá   
3    51  Mato Grosso  5103403  ...     None     None     5101   Cui

C:\Users\mathg\AppData\Local\Temp\ipykernel_5236\2698738469.py:29: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centro_x = gdf_filtrado.geometry.centroid.x.mean()
C:\Users\mathg\AppData\Local\Temp\ipykernel_5236\2698738469.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centro_y = gdf_filtrado.geometry.centroid.y.mean()
